In [69]:
import pandas as pd

df = pd.read_csv("comp1804_coursework_dataset_25-26_hansard_motions_full_revision.csv")
df.head()

,motion_id,motion_topic,policy_id,motion_text_length,speaker,num_tokens,has_entity,motion_text,has_entity_date,has_entity_person,has_entity_event_or_org,aux_numeric_feature,date,date_continuous,motion_priority,party
0,0,Asylum System - More strict,1087,162,Oliver Letwin,27,True,"Nationality, Immigration and Asylum Bill — Sup...",False,True,True,-0.513867,2002-06-11,0.227273,NaN,Con
1,1,Asylum System - More strict,1087,1844,David Blunkett,322,True,"Nationality, Immigration and Asylum Bill — App...",False,True,True,-1.059214,2002-06-11,0.227273,NaN,Lab
2,2,Asylum System - More strict,1087,115,Simon Hughes,21,True,"Nationality, Immigration and Asylum Bill — Ear...",False,True,True,-0.062679,2002-06-11,0.227273,NaN,LD
3,3,Asylum System - More strict,1087,278,Beverley Hughes,41,True,"Nationality, Immigration and Asylum Bill — Wit...",False,True,True,0.955142,2002-06-12,0.227273,NaN,Lab
4,4,Asylum System - More strict,1087,122,Simon Hughes,21,True,"Nationality, Immigration and Asylum Bill — Aut...",False,True,True,-0.985726,2002-06-12,0.227273,NaN,LD


In [70]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

Shape: (592, 16)

Missing values:
motion_id                    0
motion_topic                 0
policy_id                    0
motion_text_length           0
speaker                     71
num_tokens                   0
has_entity                   0
motion_text                  0
has_entity_date              0
has_entity_person            0
has_entity_event_or_org      0
aux_numeric_feature        103
date                         0
date_continuous              0
motion_priority            592
party                       71
dtype: int64


In [71]:
print(df['motion_topic'].value_counts())
print(df.isnull().sum())

motion_topic
European Union - For                                 128
Terrorism laws - For                                  65
Reduce Spending on Welfare Benefits                   61
More powers for local councils                        52
Further devolution to Scotland                        43
Stop climate change                                   43
Schools - Greater Autonomy                            33
Shift Powers from MPs in the Commons to Ministers     31
Asylum System - More strict                           30
Homosexuality - Equal rights                          30
Increase VAT                                          30
Identity cards - For introduction                     24
Higher taxes on alcoholic drinks                      22
Name: count, dtype: int64
motion_id                    0
motion_topic                 0
policy_id                    0
motion_text_length           0
speaker                     71
num_tokens                   0
has_entity                   0
mot

In [72]:
df['has_entity'] = df['has_entity'].fillna(0)
df['has_entity_date'] = df['has_entity_date'].fillna(0)
df['has_entity_person'] = df['has_entity_person'].fillna(0)

In [73]:
df['aux_numeric_feature'] = df['aux_numeric_feature'].fillna(df['aux_numeric_feature'].median())
df['speaker'] = df['speaker'].fillna("Unknown")
df['party'] = df['party'].fillna("Unknown")

print("Missing after cleaning:")
print(df.isnull().sum())

Missing after cleaning:
motion_id                    0
motion_topic                 0
policy_id                    0
motion_text_length           0
speaker                      0
num_tokens                   0
has_entity                   0
motion_text                  0
has_entity_date              0
has_entity_person            0
has_entity_event_or_org      0
aux_numeric_feature          0
date                         0
date_continuous              0
motion_priority            592
party                        0
dtype: int64


In [74]:
SIX_CLASSES = [
    'Terrorism laws - For',
    'Reduce Spending on Welfare Benefits',
    'More powers for local councils',
    'Further devolution to Scotland',
    'Stop climate change',
    'Schools - Greater Autonomy'
]

df_task1 = df[df['motion_topic'].isin(SIX_CLASSES)].copy()
df_task1.reset_index(drop=True, inplace=True)

print("Filtered shape:", df_task1.shape)
print(df_task1['motion_topic'].value_counts())

Filtered shape: (297, 16)
motion_topic
Terrorism laws - For                   65
Reduce Spending on Welfare Benefits    61
More powers for local councils         52
Further devolution to Scotland         43
Stop climate change                    43
Schools - Greater Autonomy             33
Name: count, dtype: int64


In [75]:
X = df_task1['motion_text']
y = df_task1['motion_topic']

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1, 3),
    sublinear_tf=True,   # log-scale TF — big boost for text models
    min_df=1
)

X_tfidf = vectorizer.fit_transform(X)
print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (297, 10000)


In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (237, 10000)
Test: (60, 10000)


In [80]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]}

grid = GridSearchCV(
    LinearSVC(max_iter=5000, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid.fit(X_train, y_train)

print("Best C:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))

Best C: {'C': 0.5}
Best CV accuracy: 0.8612


In [81]:
best_svc = grid.best_estimator_

print("Train:", round(accuracy_score(y_train, best_svc.predict(X_train)), 4))
print("Test: ", round(accuracy_score(y_test,  best_svc.predict(X_test)),  4))
print(classification_report(y_test, best_svc.predict(X_test)))

Train: 1.0
Test:  0.8167
                                     precision    recall  f1-score   support

     Further devolution to Scotland       0.90      1.00      0.95         9
     More powers for local councils       0.75      0.60      0.67        10
Reduce Spending on Welfare Benefits       0.79      0.92      0.85        12
         Schools - Greater Autonomy       0.67      0.57      0.62         7
                Stop climate change       1.00      0.67      0.80         9
               Terrorism laws - For       0.81      1.00      0.90        13

                           accuracy                           0.82        60
                          macro avg       0.82      0.79      0.80        60
                       weighted avg       0.82      0.82      0.81        60



In [82]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

# LinearSVC needs wrapping to support predict_proba for soft voting
svc_cal = CalibratedClassifierCV(LinearSVC(C=grid.best_params_['C'], max_iter=5000, random_state=42), cv=3)
lr2     = LogisticRegression(C=5, max_iter=1000, solver='lbfgs', multi_class='multinomial', random_state=42)
nb2     = MultinomialNB(alpha=0.1)

ensemble = VotingClassifier(
    estimators=[('svc', svc_cal), ('lr', lr2), ('nb', nb2)],
    voting='soft'
)
ensemble.fit(X_train, y_train)

print("Ensemble Train:", round(accuracy_score(y_train, ensemble.predict(X_train)), 4))
print("Ensemble Test: ", round(accuracy_score(y_test,  ensemble.predict(X_test)),  4))

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Ensemble Train: 0.9958
Ensemble Test:  0.8333


In [83]:
from sklearn.model_selection import cross_val_score

svc_full = LinearSVC(C=grid.best_params_['C'], max_iter=5000, random_state=42)

scores = cross_val_score(svc_full, X_tfidf, y, cv=10, scoring='accuracy')

print("10-fold CV scores:", [round(s, 4) for s in scores])
print("Mean accuracy:    ", round(scores.mean(), 4))
print("Std:              ", round(scores.std(), 4))

10-fold CV scores: [np.float64(0.8), np.float64(0.8333), np.float64(0.9), np.float64(0.7333), np.float64(0.9333), np.float64(0.8333), np.float64(0.8333), np.float64(0.8621), np.float64(0.7586), np.float64(0.6552)]
Mean accuracy:     0.8143
Std:               0.0775


In [84]:
svc_final = LinearSVC(C=grid.best_params_['C'], max_iter=5000, random_state=42)
svc_final.fit(X_tfidf, y)  # use ALL 297 samples

# Verify on training data
print("Final model trained on full dataset")
print("Train Accuracy:", round(accuracy_score(y, svc_final.predict(X_tfidf)), 4))
print("\n(Use the CV score from Cell 17 as your real performance estimate)")

Final model trained on full dataset
Train Accuracy: 1.0

(Use the CV score from Cell 17 as your real performance estimate)


In [85]:
print(y.value_counts())
print("\nSmallest classes:")
print(y.value_counts().tail(3))

motion_topic
Terrorism laws - For                   65
Reduce Spending on Welfare Benefits    61
More powers for local councils         52
Further devolution to Scotland         43
Stop climate change                    43
Schools - Greater Autonomy             33
Name: count, dtype: int64

Smallest classes:
motion_topic
Further devolution to Scotland    43
Stop climate change               43
Schools - Greater Autonomy        33
Name: count, dtype: int64


In [86]:
from sklearn.svm import LinearSVC

svc_weighted = LinearSVC(C=0.5, max_iter=5000, random_state=42, class_weight='balanced')
svc_weighted.fit(X_train, y_train)

print("Train:", round(accuracy_score(y_train, svc_weighted.predict(X_train)), 4))
print("Test: ", round(accuracy_score(y_test,  svc_weighted.predict(X_test)),  4))
print(classification_report(y_test, svc_weighted.predict(X_test)))

Train: 1.0
Test:  0.8333
                                     precision    recall  f1-score   support

     Further devolution to Scotland       0.90      1.00      0.95         9
     More powers for local councils       0.75      0.60      0.67        10
Reduce Spending on Welfare Benefits       0.85      0.92      0.88        12
         Schools - Greater Autonomy       0.62      0.71      0.67         7
                Stop climate change       1.00      0.67      0.80         9
               Terrorism laws - For       0.87      1.00      0.93        13

                           accuracy                           0.83        60
                          macro avg       0.83      0.82      0.81        60
                       weighted avg       0.84      0.83      0.83        60



In [87]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# char n-grams — catches suffixes, word fragments, spelling variations
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=20000,
    sublinear_tf=True,
    min_df=1
)

X_char = char_vectorizer.fit_transform(X)
print("Char TF-IDF shape:", X_char.shape)

# combine word + char features
X_combined = hstack([X_tfidf, X_char])
print("Combined shape:", X_combined.shape)

Char TF-IDF shape: (297, 20000)
Combined shape: (297, 30000)


In [88]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

svc_combined = LinearSVC(C=0.5, max_iter=5000, random_state=42, class_weight='balanced')
svc_combined.fit(X_train_c, y_train_c)

print("Train:", round(accuracy_score(y_train_c, svc_combined.predict(X_train_c)), 4))
print("Test: ", round(accuracy_score(y_test_c,  svc_combined.predict(X_test_c)),  4))
print(classification_report(y_test_c, svc_combined.predict(X_test_c)))

Train: 1.0
Test:  0.8667
                                     precision    recall  f1-score   support

     Further devolution to Scotland       0.90      1.00      0.95         9
     More powers for local councils       0.88      0.70      0.78        10
Reduce Spending on Welfare Benefits       0.92      0.92      0.92        12
         Schools - Greater Autonomy       0.75      0.86      0.80         7
                Stop climate change       1.00      0.67      0.80         9
               Terrorism laws - For       0.81      1.00      0.90        13

                           accuracy                           0.87        60
                          macro avg       0.88      0.86      0.86        60
                       weighted avg       0.88      0.87      0.86        60



In [89]:
from sklearn.model_selection import cross_val_score

svc_cv = LinearSVC(C=0.5, max_iter=5000, random_state=42, class_weight='balanced')
scores_combined = cross_val_score(svc_cv, X_combined, y, cv=10, scoring='accuracy')

print("10-fold CV scores:", [round(s, 4) for s in scores_combined])
print("Mean accuracy:    ", round(scores_combined.mean(), 4))
print("Std:              ", round(scores_combined.std(), 4))

10-fold CV scores: [np.float64(0.8333), np.float64(0.8667), np.float64(0.9), np.float64(0.7667), np.float64(0.9333), np.float64(0.8), np.float64(0.8), np.float64(0.8621), np.float64(0.8276), np.float64(0.7241)]
Mean accuracy:     0.8314
Std:               0.0591


In [90]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

param_grid = {'C': [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]}

grid2 = GridSearchCV(
    LinearSVC(max_iter=5000, random_state=42, class_weight='balanced'),
    param_grid,
    cv=10,
    scoring='accuracy'
)
grid2.fit(X_combined, y)

print("Best C:", grid2.best_params_)
print("Best CV accuracy:", round(grid2.best_score_, 4))

Best C: {'C': 0.1}
Best CV accuracy: 0.838


In [91]:
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier(
    loss='modified_huber',
    alpha=0.0001,
    max_iter=200,
    random_state=42,
    class_weight='balanced'
)
sgd.fit(X_train_c, y_train_c)

print("Train:", round(accuracy_score(y_train_c, sgd.predict(X_train_c)), 4))
print("Test: ", round(accuracy_score(y_test_c,  sgd.predict(X_test_c)),  4))
print(classification_report(y_test_c, sgd.predict(X_test_c)))

Train: 1.0
Test:  0.85
                                     precision    recall  f1-score   support

     Further devolution to Scotland       0.89      0.89      0.89         9
     More powers for local councils       0.78      0.70      0.74        10
Reduce Spending on Welfare Benefits       0.92      0.92      0.92        12
         Schools - Greater Autonomy       0.62      0.71      0.67         7
                Stop climate change       1.00      0.78      0.88         9
               Terrorism laws - For       0.87      1.00      0.93        13

                           accuracy                           0.85        60
                          macro avg       0.85      0.83      0.84        60
                       weighted avg       0.86      0.85      0.85        60



In [92]:
sgd_cv = SGDClassifier(
    loss='modified_huber',
    alpha=0.0001,
    max_iter=200,
    random_state=42,
    class_weight='balanced'
)
scores_sgd = cross_val_score(sgd_cv, X_combined, y, cv=10, scoring='accuracy')

print("SGD CV scores:", [round(s, 4) for s in scores_sgd])
print("Mean:         ", round(scores_sgd.mean(), 4))
print("Std:          ", round(scores_sgd.std(), 4))

SGD CV scores: [np.float64(0.8), np.float64(0.8667), np.float64(0.8667), np.float64(0.9333), np.float64(0.9333), np.float64(0.8), np.float64(0.8333), np.float64(0.8276), np.float64(0.8276), np.float64(0.7586)]
Mean:          0.8447
Std:           0.0537


In [93]:
sgd_best = SGDClassifier(
    loss='modified_huber',
    alpha=0.0001,
    max_iter=200,
    random_state=42,
    class_weight='balanced'
)
sgd_best.fit(X_train_c, y_train_c)

print("Train:", round(accuracy_score(y_train_c, sgd_best.predict(X_train_c)), 4))
print("Test: ", round(accuracy_score(y_test_c,  sgd_best.predict(X_test_c)),  4))
print(classification_report(y_test_c, sgd_best.predict(X_test_c)))

Train: 1.0
Test:  0.85
                                     precision    recall  f1-score   support

     Further devolution to Scotland       0.89      0.89      0.89         9
     More powers for local councils       0.78      0.70      0.74        10
Reduce Spending on Welfare Benefits       0.92      0.92      0.92        12
         Schools - Greater Autonomy       0.62      0.71      0.67         7
                Stop climate change       1.00      0.78      0.88         9
               Terrorism laws - For       0.87      1.00      0.93        13

                           accuracy                           0.85        60
                          macro avg       0.85      0.83      0.84        60
                       weighted avg       0.86      0.85      0.85        60



In [94]:
param_grid_sgd = {'alpha': [0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.005]}

grid_sgd = GridSearchCV(
    SGDClassifier(loss='modified_huber', max_iter=200, random_state=42, class_weight='balanced'),
    param_grid_sgd,
    cv=10,
    scoring='accuracy'
)
grid_sgd.fit(X_combined, y)

print("Best alpha:", grid_sgd.best_params_)
print("Best CV accuracy:", round(grid_sgd.best_score_, 4))

Best alpha: {'alpha': 0.0001}
Best CV accuracy: 0.8447


In [95]:
summary = {
    "LinearSVC (word only)":        0.8143,
    "LinearSVC + class_weight":     0.8314,
    "LinearSVC + char + balanced":  0.8314,
    "LinearSVC best C on combined": 0.8380,
    "SGD combined":                 0.8447,
}

print(f"{'Model':<35} {'CV Mean':>10}")
print("-" * 47)
for name, score in summary.items():
    marker = " ✅" if score == max(summary.values()) else ""
    print(f"{name:<35} {score:>10.4f}{marker}")

Model                                  CV Mean
-----------------------------------------------
LinearSVC (word only)                   0.8143
LinearSVC + class_weight                0.8314
LinearSVC + char + balanced             0.8314
LinearSVC best C on combined            0.8380
SGD combined                            0.8447 ✅
